In [1]:
!pip install ultralytics easyocr opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 31.0 MB/s eta 0:00:00


In [2]:
import cv2
import easyocr
from ultralytics import YOLO
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
All libraries imported successfully!


In [3]:
import os

os.environ['KAGGLE_USERNAME'] = "animeshkumar011"

os.environ['KAGGLE_KEY'] = "KGAT_dd697e3e1593dc08c358f6167c4419eb"

print("Success! Environment configured manually.")

Success! Environment configured manually.


In [4]:

!kaggle datasets download -d aslanahmedov/number-plate-detection

!unzip -q number-plate-detection.zip

print("Dataset downloaded and unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/aslanahmedov/number-plate-detection
License(s): copyright-authors
100% 413M/413M [00:21<00:00, 20.1MB/s] 

Dataset downloaded and unzipped successfully!


In [ ]:
!ls

In [ ]:
!ls images

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# 1. Setup paths
base_dir = 'images'
images_dir = 'datasets/images'
labels_dir = 'datasets/labels'

for split in ['train', 'val']:
    os.makedirs(os.path.join(images_dir, split), exist_ok=True)
    os.makedirs(os.path.join(labels_dir, split), exist_ok=True)

# Helper function to convert XML bounding boxes to normalized YOLO .txt format
def convert_xml_to_yolo(xml_path, txt_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Get image dimensions
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(txt_path, 'w') as f:
            for obj in root.findall('object'):
                # Assuming class 0 for license plate
                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)

                # Normalize coordinates for YOLO (0 to 1)
                x_center = ((xmin + xmax) / 2.0) / w
                y_center = ((ymin + ymax) / 2.0) / h
                width = (xmax - xmin) / w
                height = (ymax - ymin) / h

                f.write(f"0 {x_center} {y_center} {width} {height}\n")
        return True
    except Exception as e:
        return False

# 2. Gather images and split 80/20
images = [f for f in os.listdir(base_dir) if f.lower().endswith('.jpeg') or f.lower().endswith('.jpg')]
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

# 3. Process and move files
def process_split(file_list, split_type):
    for img_name in file_list:
        base_name = os.path.splitext(img_name)[0]
        xml_name = base_name + '.xml'

        src_img = os.path.join(base_dir, img_name)
        src_xml = os.path.join(base_dir, xml_name)

        if os.path.exists(src_xml):
            # Target paths
            dest_img = os.path.join(images_dir, split_type, img_name)
            dest_txt = os.path.join(labels_dir, split_type, base_name + '.txt')

            # Convert XML to TXT directly into the destination folder
            if convert_xml_to_yolo(src_xml, dest_txt):
                shutil.move(src_img, dest_img)

process_split(train_imgs, 'train')
process_split(val_imgs, 'val')

print("Data successfully split, restructured, and converted to YOLO format!")

In [ ]:
!ls


In [ ]:
yaml_content = """
train: /content/datasets/images/train
val: /content/datasets/images/val

nc: 1
names: ['license_plate']
"""

with open('data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("data.yaml updated! It now points to the correct 'datasets' folders.")

In [ ]:
from ultralytics import YOLO

# 2. Load a pre-trained YOLOv8 Nano model
# This downloads the base neural network weights optimized for speed
model = YOLO('yolov8n.pt')

# 3. Train the model on your custom license plate dataset
results = model.train(
    data='data.yaml',    # Path to our verified config file
    epochs=20,           # 20 loops through the dataset to start with
    imgsz=640,           # Standard pixel resolution for YOLOv8
    device=0,
    fliplr=0.0,   # ❌ STOPS horizontal flipping
    flipud=0.0    # ❌ STOPS vertical flipping
)

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load your newly trained custom weights
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 2. Grab a random image from the validation set
val_image_dir = 'datasets/images/val'
sample_img_name = os.listdir(val_image_dir)[0]
sample_img_path = os.path.join(val_image_dir, sample_img_name)

# 3. Run inference
results = model(sample_img_path, conf=0.15, save=True)

# 4. Plot the results (YOLO draws the bounding box automatically)
annotated_img = results[0].plot()

# 5. Display inside the notebook
img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f"Prediction Sample: {sample_img_name}")
plt.show()

In [ ]:
import cv2
import os
import easyocr
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Initialize the EasyOCR Reader (English characters and numbers)
# This will download the OCR language models on the first run
reader = easyocr.Reader(['en'], gpu=True)

# 2. Load your custom YOLO model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 3. Select a validation image
val_image_dir = 'datasets/images/val'
sample_img_name = os.listdir(val_image_dir)[0]
sample_img_path = os.path.join(val_image_dir, sample_img_name)

# 4. Run inference (using our 0.15 threshold)
results = model(sample_img_path, conf=0.15, verbose=False)
result = results[0]

# Load the raw original image using OpenCV
original_img = cv2.imread(sample_img_path)

# 5. Check if any license plates were detected
if len(result.boxes) == 0:
    print(f"No license plates detected in {sample_img_name}")
else:
    print(f"Detected {len(result.boxes)} plate(s). Processing...")

    # Loop through every detected box
    for i, box in enumerate(result.boxes):
        # Get coordinates in pixel format [xmin, ymin, xmax, ymax]
        coords = box.xyxy[0].tolist()
        xmin, ymin, xmax, ymax = map(int, coords)
        padding_x = int((xmax - xmin) * 0.15)  # Add 15% width to the sides
        padding_y = int((ymax - ymin) * 0.15)  # Add 10% height to top/bottom

        # Constrain the padded coordinates so they don't go outside the image frame
        xmin = max(0, xmin - padding_x)
        ymin = max(0, ymin - padding_y)
        xmax = min(original_img.shape[1], xmax + padding_x)
        ymax = min(original_img.shape[0], ymax + padding_y)
        # PART A: CROP THE LICENSE PLATE
        # Slice the numpy array: [height_range, width_range]
        cropped_plate = original_img[ymin:ymax, xmin:xmax]

        # Save the cropped plate to your disk
        crop_filename = f"cropped_plate_{i}.jpeg"
        cv2.imwrite(crop_filename, cropped_plate)

        # PART B: READ THE CHARACTERS
        # Convert BGR to RGB for EasyOCR and Matplotlib
        cropped_rgb = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB)
        ocr_results = reader.readtext(cropped_rgb)

        # Extract and clean up the text strings found by OCR
        text_outputs = [text[1] for text in ocr_results]
        final_plate_text = " ".join(text_outputs).strip().upper()

        # PART C: VISUALIZE RESULTS
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Display original image with the bounding box
        annotated_img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        axes[0].imshow(annotated_img)
        axes[0].set_title("1. YOLO Detection Box")
        axes[0].axis('off')

        # Display the cropped isolated plate patch
        axes[1].imshow(cropped_rgb)
        axes[1].set_title(f"2. Cropped Segment\nOCR Read: {final_plate_text if final_plate_text else 'No text read'}")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()
        print(f"📄 Raw Text Extracted from Plate {i+1}: '{final_plate_text}'")

In [ ]:
import cv2
from ultralytics import YOLO
import easyocr

# 1. Initialize your core models
model = YOLO('/content/runs/detect/train/weights/best.pt')
reader = easyocr.Reader(['en'], gpu=True)

# 2. Setup your video stream file path
video_filename = 'traffic_video.mp4'
cap = cv2.VideoCapture(video_filename)

if not cap.isOpened():
    print(f" Error: Could not open video file '{video_filename}'.")
else:
    print(f" Processing frames from '{video_filename}'...")

# Gather video metadata properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30

# Setup video writer to output our visualized tracking stream
out = cv2.VideoWriter('output_tracked.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

# Pure Python dictionary memory to map: Tracking ID -> Read Text
tracked_plates = {}
frame_count = 0

# 3. Main Processing Frame Loop
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1

    # Run YOLOv8 in tracking mode
    results = model.track(frame, conf=0.15, persist=True, verbose=False)
    result = results[0]

    # If tracks exist in the current frame matrix
    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.tolist()
        track_ids = result.boxes.id.int().tolist()

        for box, track_id in zip(boxes, track_ids):
            xmin, ymin, xmax, ymax = map(int, box)

            # THE MEMORY CHECK: If tracking ID is already cached, bypass OCR
            if track_id in tracked_plates:
                final_text = tracked_plates[track_id]
            else:
                # New vehicle tracking ID detected! Run crop & EasyOCR
                padding_x = int((xmax - xmin) * 0.15)
                padding_y = int((ymax - ymin) * 0.15)

                xmin = max(0, xmin - padding_x)
                ymin = max(0, ymin - padding_y)
                xmax = min(frame_width, xmax + padding_x)
                ymax = min(frame_height, ymax + padding_y)

                cropped_plate = frame[ymin:ymax, xmin:xmax]

                if cropped_plate.size > 0:
                    ocr_results = reader.readtext(cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB))
                    text_outputs = [text[1] for text in ocr_results]
                    final_text = " ".join(text_outputs).strip().upper()

                    if not final_text:
                        final_text = "UNKNOWN"

                    # Store text strings strictly inside your local Python runtime dictionary memory
                    tracked_plates[track_id] = final_text
                    print(f" [Frame {frame_count}] Captured New Plate: ID {track_id} ➔ {final_text}")

            # Draw visual bounding box overlays onto the video output channel frame
            cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(frame, f"ID {track_id}: {final_text}", (xmin, ymin - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Save frame step to output stream
    out.write(frame)

# Safely close file pointers and release memory blocks
cap.release()
out.release()

print(f"\n Done! Total processed frames: {frame_count}")
print("Check out your clean visualization output at 'output_tracked.mp4'")

In [ ]:
# 1. Install Gradio cleanly
!pip install gradio -q

import gradio as gr
import cv2
import numpy as np
import easyocr
from ultralytics import YOLO
import tempfile

# 2. Initialize Models Once in Memory
model = YOLO('/content/runs/detect/train/weights/best.pt')
reader = easyocr.Reader(['en'], gpu=True)

# =====================================================================
# CORE FUNCTION A: PROCESS SINGLE IMAGE UPLOADS
# =====================================================================
def process_single_image(input_image):
    if input_image is None:
        return None, "Please upload an image."

    # Gradio works in RGB; convert to BGR for OpenCV
    original_img = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)

    # Run YOLO using a slightly larger input size (imgsz=1280) to capture distant plates
    results = model(original_img, imgsz=1280, conf=0.10, verbose=False)
    result = results[0]

    if len(result.boxes) == 0:
        return cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB), "No license plate detected."

    # Isolate first detected plate
    box = result.boxes[0]
    coords = box.xyxy[0].tolist()
    xmin, ymin, xmax, ymax = map(int, coords)

    # Apply your exact Padding Trick
    padding_x = int((xmax - xmin) * 0.15)
    padding_y = int((ymax - ymin) * 0.15)
    xmin = max(0, xmin - padding_x)
    ymin = max(0, ymin - padding_y)
    xmax = min(original_img.shape[1], xmax + padding_x)
    ymax = min(original_img.shape[0], ymax + padding_y)

    # Crop and OCR translation
    cropped_plate = original_img[ymin:ymax, xmin:xmax]
    cropped_rgb = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB)

    ocr_results = reader.readtext(cropped_rgb)
    text_outputs = [text[1] for text in ocr_results]
    final_text = " ".join(text_outputs).strip().upper()

    # Paint visualization onto the original frame array
    cv2.rectangle(original_img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)
    cv2.putText(original_img, final_text, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    return cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB), final_text if final_text else "COULD NOT READ TEXT"


# =====================================================================
# CORE FUNCTION B: PROCESS VIDEO STREAMS (WITH TRACKING & HIGH RES)
# =====================================================================
def process_video_stream(video_path):
    if not video_path:
        return None

    cap = cv2.VideoCapture(video_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30

    # Create a temporary secure path to save output file
    temp_output = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
    out = cv2.VideoWriter(temp_output.name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    tracked_plates = {}

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Use imgsz=1280 and drop conf=0.10 to capture small/distant plates in video
        results = model.track(frame, imgsz=1280, conf=0.10, persist=True, verbose=False)
        result = results[0]

        if result.boxes.id is not None:
            boxes = result.boxes.xyxy.tolist()
            track_ids = result.boxes.id.int().tolist()

            for box, track_id in zip(boxes, track_ids):
                xmin, ymin, xmax, ymax = map(int, box)

                # Dictionary memory gatekeeper check to remove redundancies
                if track_id in tracked_plates:
                    final_text = tracked_plates[track_id]
                else:
                    padding_x = int((xmax - xmin) * 0.15)
                    padding_y = int((ymax - ymin) * 0.15)
                    xmin = max(0, xmin - padding_x)
                    ymin = max(0, ymin - padding_y)
                    xmax = min(frame_width, xmax + padding_x)
                    ymax = min(frame_height, ymax + padding_y)

                    cropped_plate = frame[ymin:ymax, xmin:xmax]
                    if cropped_plate.size > 0:
                        ocr_results = reader.readtext(cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB))
                        text_outputs = [text[1] for text in ocr_results]
                        final_text = " ".join(text_outputs).strip().upper()
                        tracked_plates[track_id] = final_text if final_text else "UNKNOWN"
                    else:
                        final_text = "UNKNOWN"

                # Step 6: Draw layout on video frame matrix
                cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
                cv2.putText(frame, f"ID {track_id}: {final_text}", (xmin, ymin - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()
    return temp_output.name

# =====================================================================
# STEP 2: BUILD THE GRADIO MULTI-TAB DASHBOARD UI
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚗 Automated License Plate Recognition (ALPR) System Dashboard")
    gr.Markdown("Powered by YOLOv8 Custom Detector (tuned for small objects at 1280px inference resolution) and EasyOCR.")

    with gr.Tabs():
        # TAB 1: Single Image Processor
        with gr.TabItem("🖼️ Image Detection"):
            with gr.Row():
                img_input = gr.Image(label="Upload Vehicle Image")
                with gr.Column():
                    img_output = gr.Image(label="Processed Result View")
                    text_output = gr.Textbox(label="Extracted Plate Text String")
            img_button = gr.Button("Analyze Frame Image")
            img_button.click(process_single_image, inputs=img_input, outputs=[img_output, text_output])

        # TAB 2: Live Video Tracker File Engine
        with gr.TabItem("📹 Video Tracking Stream"):
            gr.Markdown("⚠️ *Processing videos frame-by-frame with 1280px optimization matrices can take a minute depending on duration.*")
            with gr.Row():
                video_input = gr.Video(label="Upload Traffic MP4 File")
                video_output = gr.Video(label="Processed Annotated Output Stream")
            video_button = gr.Button("Process & Track Video Sequence")
            video_button.click(process_video_stream, inputs=video_input, outputs=video_output)

# 3. Launch App and expose public tunnel URL
demo.launch(share=True, debug=True)